In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path

class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        # Use only valid images identified during preprocessing
        self.img_names = [f.stem for f in self.mask_dir.glob("*.png")]
        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = self.img_dir / f"{self.img_names[idx]}.jpg"
        mask_path = self.mask_dir / f"{self.img_names[idx]}.png"
        
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path) # Single channel category IDs
        
        if self.transform:
            image = self.transform(image)
            # Convert mask to tensor without normalization
            mask = torch.as_tensor(np.array(mask), dtype=torch.long)
        
        return image, mask

# Transforms as expected by ResNet backbones
transform = transforms.Compose([
    transforms.Resize((520, 520)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = COCOSegmentationDataset("/kaggle/input/coco2017/train2017", "/kaggle/working/masks/train", transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet101

def get_model(num_classes=21):
    # Set weights=None for training from scratch
    model = deeplabv3_resnet101(weights=None, num_classes=num_classes)
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_model(num_classes=81).to(device)

In [ ]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

for epoch in range(num_epochs):
    model.train()
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)['out'] # DeepLabV3 returns a dict
        
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
    print(f"Epoch {epoch+1} Loss: {loss.item()}")